In [1]:
import sys
#sys.path.append('../input/iterativestratification')
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

In [2]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import os
import copy
import seaborn as sns

from sklearn import preprocessing
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import warnings
warnings.filterwarnings('ignore')

In [3]:
from sklearn.preprocessing import QuantileTransformer

In [4]:
train_features = pd.read_csv('../lish-moa/train_features.csv')
train_targets_scored = pd.read_csv('../lish-moa/train_targets_scored.csv')
train_targets_nonscored = pd.read_csv('../lish-moa/train_targets_nonscored.csv')
test_features = pd.read_csv('../lish-moa/test_features.csv')
sample_submission = pd.read_csv('../lish-moa/sample_submission.csv')

In [5]:
GENES = [col for col in train_features.columns if col.startswith('g-')]
CELLS = [col for col in train_features.columns if col.startswith('c-')]

In [6]:
#RankGauss

for col in (GENES + CELLS):

    transformer = QuantileTransformer(n_quantiles=100,random_state=0, output_distribution="normal")
    vec_len = len(train_features[col].values)
    vec_len_test = len(test_features[col].values)
    raw_vec = train_features[col].values.reshape(vec_len, 1)
    transformer.fit(raw_vec)

    train_features[col] = transformer.transform(raw_vec).reshape(1, vec_len)[0]
    test_features[col] = transformer.transform(test_features[col].values.reshape(vec_len_test, 1)).reshape(1, vec_len_test)[0]

In [7]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(seed=42)

In [8]:
# GENES
n_comp_GENES = xx
g_pca = PCA(n_components=n_comp_GENES, random_state=42)
train_features = pd.concat((train_features, pd.DataFrame(g_pca.fit_transform(train_features[GENES]), columns=[f'pca_G-{i}' for i in range(n_comp_GENES)])), axis=1)
test_features = pd.concat((test_features, pd.DataFrame(g_pca.transform(test_features[GENES]), columns=[f'pca_G-{i}' for i in range(n_comp_GENES)])), axis=1)

In [9]:
#CELLS
n_comp_CELLS = xx
c_pca = PCA(n_components=n_comp_CELLS, random_state=42)
train_features = pd.concat((train_features, pd.DataFrame(c_pca.fit_transform(train_features[CELLS]), columns=[f'pca_C-{i}' for i in range(n_comp_CELLS)])), axis=1)
test_features = pd.concat((test_features, pd.DataFrame(c_pca.transform(test_features[CELLS]), columns=[f'pca_C-{i}' for i in range(n_comp_CELLS)])), axis=1)

In [10]:
train_features.shape

(23814, 1493)

In [11]:
test_features.shape

(3982, 1493)

In [12]:
from sklearn.feature_selection import VarianceThreshold


var_thresh = VarianceThreshold(xx)  
train_features_transformed = var_thresh.fit_transform(train_features.iloc[:, 4:])

test_features_transformed =var_thresh.transform(test_features.iloc[:, 4:])


train_features = pd.DataFrame(train_features[['sig_id','cp_type','cp_time','cp_dose']].values.reshape(-1, 4),\
                              columns=['sig_id','cp_type','cp_time','cp_dose'])

train_features = pd.concat([train_features, pd.DataFrame(train_features_transformed)], axis=1)


test_features = pd.DataFrame(test_features[['sig_id','cp_type','cp_time','cp_dose']].values.reshape(-1, 4),\
                             columns=['sig_id','cp_type','cp_time','cp_dose'])

test_features = pd.concat([test_features, pd.DataFrame(test_features_transformed)], axis=1)

In [13]:
train_features.shape

(23814, 1030)

In [14]:
test_features.shape

(3982, 1030)

In [15]:
from sklearn.cluster import KMeans
from sklearn.cluster import KMeans
def fe_cluster(train, test,n_clusters_g =xx, n_clusters_c =xx, kind_g = 'g', kind_c = 'c',SEED = 123):
    features_g = list(train.columns[4:776])
    features_c = list(train.columns[776:876])
    #gen
    kmeans1 = KMeans(n_clusters = n_clusters_g,n_jobs = -1, random_state = SEED).fit(train[features_g])
    train[f'clusters_{kind_g}'] = kmeans1.labels_
    train = pd.get_dummies(train, columns = [f'clusters_{kind_g}'])
    test[f'clusters_{kind_g}'] = kmeans1.predict(test[features_g])
    test = pd.get_dummies(test, columns = [f'clusters_{kind_g}'])
    
    
    #cell
    kmeans1 = KMeans(n_clusters = n_clusters_c,n_jobs = -1, random_state = SEED).fit(train[features_c])
    train = pd.get_dummies(train, columns = [f'clusters_{kind_c}'])
    test[f'clusters_{kind_c}'] = kmeans1.predict(test[features_c])
    test = pd.get_dummies(test, columns = [f'clusters_{kind_c}'])
    
    return train, test
        
train_features ,test_features=fe_cluster(train_features,test_features)

[ 2 28  0 ... 15  7 15]


(23814, 1069)

In [17]:
train_features.shape

(23814, 1069)

In [18]:
test_features.shape

(3982, 1069)

In [19]:
def fe_stats(train, test):
    
    features_g = list(train.columns[4:776])
    features_c = list(train.columns[776:876])
    
    for df in train, test:
        df['g_sum'] = df[features_g].sum(axis = 1)
        df['g_mean'] = df[features_g].mean(axis = 1)
        df['g_std'] = df[features_g].std(axis = 1)
        df['g_kurt'] = df[features_g].kurtosis(axis = 1) 
        df['g_skew'] = df[features_g].skew(axis = 1)
        df['c_sum'] = df[features_c].sum(axis = 1)
        df['c_mean'] = df[features_c].mean(axis = 1)
        df['c_std'] = df[features_c].std(axis = 1)
        df['c_kurt'] = df[features_c].kurtosis(axis = 1)
        df['c_skew'] = df[features_c].skew(axis = 1)
        df['gc_sum'] = df[features_g + features_c].sum(axis = 1)
        df['gc_mean'] = df[features_g + features_c].mean(axis = 1)
        df['gc_std'] = df[features_g + features_c].std(axis = 1)
        df['gc_kurt'] = df[features_g + features_c].kurtosis(axis = 1)
        df['gc_skew'] = df[features_g + features_c].skew(axis = 1)
        
    return train, test

train_features,test_features=fe_stats(train_features,test_features)

In [20]:
train_features.shape

(23814, 1084)

In [21]:
test_features.shape

(3982, 1084)

In [22]:
train = train_features.merge(train_targets_scored, on='sig_id')
train = train[train['cp_type']!='ctl_vehicle'].reset_index(drop=True)
test = test_features[test_features['cp_type']!='ctl_vehicle'].reset_index(drop=True)
target = train[train_targets_scored.columns]

In [23]:
train = train.drop('cp_type', axis=1)
test = test.drop('cp_type', axis=1)

In [24]:
target_cols = target.drop('sig_id', axis=1).columns.values.tolist()

In [25]:
folds = train.copy()

mskf = MultilabelStratifiedKFold(n_splits=xx)

for f, (t_idx, v_idx) in enumerate(mskf.split(X=train, y=target)):
    folds.loc[v_idx, 'kfold'] = int(f)

folds['kfold'] = folds['kfold'].astype(int)

In [26]:
print(train.shape)
print(folds.shape)
print(test.shape)
print(target.shape)
print(sample_submission.shape)

(21948, 1289)
(21948, 1290)
(3624, 1083)
(21948, 207)
(3982, 207)


In [27]:
class MoADataset:
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets
        
    def __len__(self):
        return (self.features.shape[0])
    
    def __getitem__(self, idx):
        dct = {
            'x' : torch.tensor(self.features[idx, :], dtype=torch.float),
            'y' : torch.tensor(self.targets[idx, :], dtype=torch.float)            
        }
        return dct
    
class TestDataset:
    def __init__(self, features):
        self.features = features
        
    def __len__(self):
        return (self.features.shape[0])
    
    def __getitem__(self, idx):
        dct = {
            'x' : torch.tensor(self.features[idx, :], dtype=torch.float)
        }
        return dct

In [28]:
def train_fn(model, optimizer, scheduler, loss_fn, dataloader, device):
    model.train()
    final_loss = 0
    
    for data in dataloader:
        optimizer.zero_grad()
        inputs, targets = data['x'].to(device), data['y'].to(device)
#         print(inputs.shape)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        final_loss += loss.item()
        
    final_loss /= len(dataloader)
    
    return final_loss


def valid_fn(model, loss_fn, dataloader, device):
    model.eval()
    final_loss = 0
    valid_preds = []
    
    for data in dataloader:
        inputs, targets = data['x'].to(device), data['y'].to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, targets)
        
        final_loss += loss.item()
        valid_preds.append(outputs.sigmoid().detach().cpu().numpy())
        
    final_loss /= len(dataloader)
    valid_preds = np.concatenate(valid_preds)
    
    return final_loss, valid_preds

def inference_fn(model, dataloader, device):
    model.eval()
    preds = []
    
    for data in dataloader:
        inputs = data['x'].to(device)

        with torch.no_grad():
            outputs = model(inputs)
        
        preds.append(outputs.sigmoid().detach().cpu().numpy())
        
    preds = np.concatenate(preds)
    
    return preds

In [29]:
import torch
from torch.nn.modules.loss import _WeightedLoss
import torch.nn.functional as F

class SmoothBCEwLogits(_WeightedLoss):
    def __init__(self, weight=None, reduction='mean', smoothing=0.0):
        super().__init__(weight=weight, reduction=reduction)
        self.smoothing = smoothing
        self.weight = weight
        self.reduction = reduction

    @staticmethod
    def _smooth(targets:torch.Tensor, n_labels:int, smoothing=0.0):
        assert 0 <= smoothing < 1
        with torch.no_grad():
            targets = targets * (1.0 - smoothing) + 0.5 * smoothing
        return targets

    def forward(self, inputs, targets):
        targets = SmoothBCEwLogits._smooth(targets, inputs.size(-1),
            self.smoothing)
        loss = F.binary_cross_entropy_with_logits(inputs, targets,self.weight)

        if  self.reduction == 'sum':
            loss = loss.sum()
        elif  self.reduction == 'mean':
            loss = loss.mean()

        return loss

In [30]:
class Model(nn.Module):
    def __init__(self, num_features, num_targets, hidden_size):
        super(Model, self).__init__()
        self.batch_norm1 = nn.BatchNorm1d(num_features)
        self.dense1 = nn.utils.weight_norm(nn.Linear(num_features, hidden_size))
        
        self.batch_norm2 = nn.BatchNorm1d(hidden_size)
        self.dropout2 = nn.Dropout(0.2) 
        self.dense2 = nn.utils.weight_norm(nn.Linear(hidden_size, hidden_size))
        
        self.batch_norm3 = nn.BatchNorm1d(hidden_size)
        self.dropout3 = nn.Dropout(0.2) 
        self.dense3 = nn.utils.weight_norm(nn.Linear(hidden_size, num_targets))
        
    def recalibrate_layer(self, layer):

        if(torch.isnan(layer.weight_v).sum() > 0):
            print ('recalibrate layer.weight_v')
            layer.weight_v = torch.nn.Parameter(torch.where(torch.isnan(layer.weight_v), torch.zeros_like(layer.weight_v), layer.weight_v))
            layer.weight_v = torch.nn.Parameter(layer.weight_v + 1e-7)

        if(torch.isnan(layer.weight).sum() > 0):
            print ('recalibrate layer.weight')
            layer.weight = torch.where(torch.isnan(layer.weight), torch.zeros_like(layer.weight), layer.weight)
            layer.weight += 1e-7
    
    def forward(self, x):
        x = self.batch_norm1(x)
        self.recalibrate_layer(self.dense1)
        x = F.leaky_relu(self.dense1(x))
        
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        self.recalibrate_layer(self.dense2)
        x = F.leaky_relu(self.dense2(x))#update
        
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        self.recalibrate_layer(self.dense3)
        x = self.dense3(x)
        
        return x
    
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.0, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))

In [31]:
def process_data(data):
    data = pd.get_dummies(data, columns=['cp_time','cp_dose'])
    return data

In [32]:
feature_cols = [c for c in process_data(folds).columns if c not in target_cols]
feature_cols = [c for c in feature_cols if c not in ['kfold','sig_id']]
len(feature_cols)

1085

In [33]:
# HyperParameters

DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = xx
BATCH_SIZE = xx
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NFOLDS = xx         
EARLY_STOPPING_STEPS = 10
EARLY_STOP = False

num_features=len(feature_cols)
num_targets=len(target_cols)
hidden_size=xx 

In [34]:
def run_training(fold, seed):
    seed_everything(seed)
    train = process_data(folds)
    test_ = process_data(test)
    trn_idx = train[train['kfold'] != fold].index
    val_idx = train[train['kfold'] == fold].index
    train_df = train[train['kfold'] != fold].reset_index(drop=True)
    valid_df = train[train['kfold'] == fold].reset_index(drop=True)
    x_train, y_train  = train_df[feature_cols].values, train_df[target_cols].values
    x_valid, y_valid =  valid_df[feature_cols].values, valid_df[target_cols].values
    train_dataset = MoADataset(x_train, y_train)
    valid_dataset = MoADataset(x_valid, y_valid)
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    validloader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    model = Model(
        num_features=num_features,
        num_targets=num_targets,
        hidden_size=hidden_size,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer=optimizer, pct_start=0.1, div_factor=1e3,
                                              max_lr=1e-2, epochs=EPOCHS, steps_per_epoch=len(trainloader))
    loss_fn = nn.BCEWithLogitsLoss()
    loss_tr = SmoothBCEwLogits(smoothing =0.001)
    early_stopping_steps = EARLY_STOPPING_STEPS
    early_step = 0
    oof = np.zeros((len(train), target.iloc[:, 1:].shape[1]))
    best_loss = np.inf
    for epoch in range(EPOCHS):
        train_loss = train_fn(model, optimizer,scheduler, loss_tr, trainloader, DEVICE)
        print(f"FOLD: {fold}, EPOCH: {epoch}, train_loss: {train_loss}")
        valid_loss, valid_preds = valid_fn(model, loss_fn, validloader, DEVICE)
        print(f"FOLD: {fold}, EPOCH: {epoch}, valid_loss: {valid_loss}")
        if valid_loss < best_loss:
            best_loss = valid_loss
            oof[val_idx] = valid_preds
            torch.save(model.state_dict(), f"SEED{seed}_FOLD{fold}_.pth")
        elif(EARLY_STOP == True):
            early_step += 1
            if (early_step >= early_stopping_steps):
                break
    #--------------------- PREDICTION---------------------
    x_test = test_[feature_cols].values
    testdataset = TestDataset(x_test)
    testloader = torch.utils.data.DataLoader(testdataset, batch_size=BATCH_SIZE, shuffle=False)
    model = Model(
        num_features=num_features,
        num_targets=num_targets,
        hidden_size=hidden_size,
    )
    model.load_state_dict(torch.load(f"SEED{seed}_FOLD{fold}_.pth"))
    model.to(DEVICE)
    predictions = np.zeros((len(test_), target.iloc[:, 1:].shape[1]))
    predictions = inference_fn(model, testloader, DEVICE)
    return oof, predictions

In [35]:
def run_k_fold(NFOLDS, seed):
    oof = np.zeros((len(train), len(target_cols)))
    predictions = np.zeros((len(test), len(target_cols)))
    
    for fold in range(NFOLDS):
        oof_, pred_ = run_training(fold, seed)
        
        predictions += pred_ / NFOLDS
        oof += oof_
        
    return oof, predictions

In [36]:
# Averaging on multiple SEEDS
SEED =[xx]
oof = np.zeros((len(train), len(target_cols)))
predictions = np.zeros((len(test), len(target_cols)))

for seed in SEED:
    
    oof_, predictions_ = run_k_fold(NFOLDS, seed)
    oof += oof_ / len(SEED)
    predictions += predictions_ / len(SEED)

train[target_cols] = oof

FOLD: 0, EPOCH: 0, train_loss: 0.7076065272856982
FOLD: 0, EPOCH: 0, valid_loss: 0.5841095646222433
FOLD: 0, EPOCH: 1, train_loss: 0.18544726491643068
FOLD: 0, EPOCH: 1, valid_loss: 0.02324800793495443
FOLD: 0, EPOCH: 2, train_loss: 0.02395542746839615
FOLD: 0, EPOCH: 2, valid_loss: 0.01973386626276705
FOLD: 0, EPOCH: 3, train_loss: 0.02192994622656932
FOLD: 0, EPOCH: 3, valid_loss: 0.01948953668276469
FOLD: 0, EPOCH: 4, train_loss: 0.02160073150522434
FOLD: 0, EPOCH: 4, valid_loss: 0.018828154645032354
FOLD: 0, EPOCH: 5, train_loss: 0.02040471153286023
FOLD: 0, EPOCH: 5, valid_loss: 0.017487357060114544
FOLD: 0, EPOCH: 6, train_loss: 0.019859637945699386
FOLD: 0, EPOCH: 6, valid_loss: 0.01820178495513068
FOLD: 0, EPOCH: 7, train_loss: 0.01976940331932826
FOLD: 0, EPOCH: 7, valid_loss: 0.017748731498916943
FOLD: 0, EPOCH: 8, train_loss: 0.019807905388566163
FOLD: 0, EPOCH: 8, valid_loss: 0.01718044922583633
FOLD: 0, EPOCH: 9, train_loss: 0.01966703537469491
FOLD: 0, EPOCH: 9, valid_los

FOLD: 1, EPOCH: 28, valid_loss: 0.016454293599559203
FOLD: 1, EPOCH: 29, train_loss: 0.019110853258424845
FOLD: 1, EPOCH: 29, valid_loss: 0.016337953300939664
FOLD: 1, EPOCH: 30, train_loss: 0.018939617280967724
FOLD: 1, EPOCH: 30, valid_loss: 0.01636060006502602
FOLD: 1, EPOCH: 31, train_loss: 0.018824351402238395
FOLD: 1, EPOCH: 31, valid_loss: 0.016282306466665532
FOLD: 1, EPOCH: 32, train_loss: 0.018731683134459533
FOLD: 1, EPOCH: 32, valid_loss: 0.016091340221464634
FOLD: 1, EPOCH: 33, train_loss: 0.018648449283761855
FOLD: 1, EPOCH: 33, valid_loss: 0.016186076837281387
FOLD: 1, EPOCH: 34, train_loss: 0.01845873307245664
FOLD: 1, EPOCH: 34, valid_loss: 0.016185169729093712
FOLD: 1, EPOCH: 35, train_loss: 0.018230987497820303
FOLD: 1, EPOCH: 35, valid_loss: 0.016091787877182167
FOLD: 1, EPOCH: 36, train_loss: 0.01804704125970602
FOLD: 1, EPOCH: 36, valid_loss: 0.01607039270715581
FOLD: 1, EPOCH: 37, train_loss: 0.01788341645628978
FOLD: 1, EPOCH: 37, valid_loss: 0.01616004792352517

FOLD: 3, EPOCH: 6, train_loss: 0.019914963951286595
FOLD: 3, EPOCH: 6, valid_loss: 0.017720670956704352
FOLD: 3, EPOCH: 7, train_loss: 0.019844597802521326
FOLD: 3, EPOCH: 7, valid_loss: 0.017104032242463693
FOLD: 3, EPOCH: 8, train_loss: 0.019755226130095813
FOLD: 3, EPOCH: 8, valid_loss: 0.017241086914307542
FOLD: 3, EPOCH: 9, train_loss: 0.019671361869535386
FOLD: 3, EPOCH: 9, valid_loss: 0.016827645815081067
FOLD: 3, EPOCH: 10, train_loss: 0.019730565854563162
FOLD: 3, EPOCH: 10, valid_loss: 0.017177397695680458
FOLD: 3, EPOCH: 11, train_loss: 0.019840682832858503
FOLD: 3, EPOCH: 11, valid_loss: 0.01694433204829693
FOLD: 3, EPOCH: 12, train_loss: 0.019746365407720592
FOLD: 3, EPOCH: 12, valid_loss: 0.016896755745013554
FOLD: 3, EPOCH: 13, train_loss: 0.019726589226569884
FOLD: 3, EPOCH: 13, valid_loss: 0.017225210554897785
FOLD: 3, EPOCH: 14, train_loss: 0.019847932271659374
FOLD: 3, EPOCH: 14, valid_loss: 0.017374512635999255
FOLD: 3, EPOCH: 15, train_loss: 0.019912903698591087
FO

FOLD: 4, EPOCH: 34, train_loss: 0.01834847904646244
FOLD: 4, EPOCH: 34, valid_loss: 0.016270810634725623
FOLD: 4, EPOCH: 35, train_loss: 0.018086734180076
FOLD: 4, EPOCH: 35, valid_loss: 0.016202105312711663
FOLD: 4, EPOCH: 36, train_loss: 0.017863866323843982
FOLD: 4, EPOCH: 36, valid_loss: 0.016309314200447664
FOLD: 4, EPOCH: 37, train_loss: 0.017712479361738913
FOLD: 4, EPOCH: 37, valid_loss: 0.016227689054277208
recalibrate layer.weight_v
FOLD: 4, EPOCH: 38, train_loss: 0.017450285812792104
FOLD: 4, EPOCH: 38, valid_loss: 0.01617917853097121
FOLD: 4, EPOCH: 39, train_loss: 0.017065628001896236
FOLD: 4, EPOCH: 39, valid_loss: 0.01617818646546867
FOLD: 4, EPOCH: 40, train_loss: 0.016881706706510905
FOLD: 4, EPOCH: 40, valid_loss: 0.0161812219561802
FOLD: 4, EPOCH: 41, train_loss: 0.016843906436593104
FOLD: 4, EPOCH: 41, valid_loss: 0.016241966643267207
FOLD: 4, EPOCH: 42, train_loss: 0.01671629211602685
FOLD: 4, EPOCH: 42, valid_loss: 0.01618784986850288
FOLD: 4, EPOCH: 43, train_los

FOLD: 6, EPOCH: 12, train_loss: 0.01974528781974163
FOLD: 6, EPOCH: 12, valid_loss: 0.01724473676747746
FOLD: 6, EPOCH: 13, train_loss: 0.019715457080075376
FOLD: 6, EPOCH: 13, valid_loss: 0.01806580668522252
FOLD: 6, EPOCH: 14, train_loss: 0.019770380443869494
FOLD: 6, EPOCH: 14, valid_loss: 0.01714962762263086
FOLD: 6, EPOCH: 15, train_loss: 0.01970041163552266
FOLD: 6, EPOCH: 15, valid_loss: 0.017388649905721348
FOLD: 6, EPOCH: 16, train_loss: 0.019670897641052038
FOLD: 6, EPOCH: 16, valid_loss: 0.01726059801876545
FOLD: 6, EPOCH: 17, train_loss: 0.01973912915071616
FOLD: 6, EPOCH: 17, valid_loss: 0.017547002890043788
FOLD: 6, EPOCH: 18, train_loss: 0.019795561663042277
FOLD: 6, EPOCH: 18, valid_loss: 0.017385264858603477
FOLD: 6, EPOCH: 19, train_loss: 0.019621228608183373
FOLD: 6, EPOCH: 19, valid_loss: 0.017202270734641287
FOLD: 6, EPOCH: 20, train_loss: 0.019776966255635787
FOLD: 6, EPOCH: 20, valid_loss: 0.017340944666001532
FOLD: 6, EPOCH: 21, train_loss: 0.01965063005590286
F

...

FOLD: 2, EPOCH: 32, train_loss: 0.018600517549575903
FOLD: 2, EPOCH: 32, valid_loss: 0.016649326620002586
FOLD: 2, EPOCH: 33, train_loss: 0.018476730045408774
FOLD: 2, EPOCH: 33, valid_loss: 0.01659186774243911
recalibrate layer.weight_v
FOLD: 2, EPOCH: 34, train_loss: 0.01829082354043539
FOLD: 2, EPOCH: 34, valid_loss: 0.01644261336574952
FOLD: 2, EPOCH: 35, train_loss: 0.017763183798449926
FOLD: 2, EPOCH: 35, valid_loss: 0.016455712935162917
FOLD: 2, EPOCH: 36, train_loss: 0.017645840127116594
FOLD: 2, EPOCH: 36, valid_loss: 0.01645600288692448
FOLD: 2, EPOCH: 37, train_loss: 0.017494900056566946
FOLD: 2, EPOCH: 37, valid_loss: 0.016466351329452462
FOLD: 2, EPOCH: 38, train_loss: 0.01739053563095438
FOLD: 2, EPOCH: 38, valid_loss: 0.01640861253771517
FOLD: 2, EPOCH: 39, train_loss: 0.01733366875216747
FOLD: 2, EPOCH: 39, valid_loss: 0.016348062083125114
FOLD: 2, EPOCH: 40, train_loss: 0.01723483443642274
FOLD: 2, EPOCH: 40, valid_loss: 0.016409370427330334
FOLD: 2, EPOCH: 41, train_l

FOLD: 4, EPOCH: 10, train_loss: 0.0197077111269419
FOLD: 4, EPOCH: 10, valid_loss: 0.01737944305770927
FOLD: 4, EPOCH: 11, train_loss: 0.019738621197831936
FOLD: 4, EPOCH: 11, valid_loss: 0.017619924205872748
FOLD: 4, EPOCH: 12, train_loss: 0.019774337323048174
FOLD: 4, EPOCH: 12, valid_loss: 0.017451480444934633
FOLD: 4, EPOCH: 13, train_loss: 0.019837627975413434
FOLD: 4, EPOCH: 13, valid_loss: 0.017272128206160333
FOLD: 4, EPOCH: 14, train_loss: 0.019795408400778588
FOLD: 4, EPOCH: 14, valid_loss: 0.0172642862631215
FOLD: 4, EPOCH: 15, train_loss: 0.019775797469684712
FOLD: 4, EPOCH: 15, valid_loss: 0.017454591683215566
FOLD: 4, EPOCH: 16, train_loss: 0.019863663480067864
FOLD: 4, EPOCH: 16, valid_loss: 0.016966041798392933
FOLD: 4, EPOCH: 17, train_loss: 0.019852399563369077
FOLD: 4, EPOCH: 17, valid_loss: 0.017111794609162543
FOLD: 4, EPOCH: 18, train_loss: 0.019800758825089686
FOLD: 4, EPOCH: 18, valid_loss: 0.017014971002936363
FOLD: 4, EPOCH: 19, train_loss: 0.01970532285765959

FOLD: 5, EPOCH: 37, train_loss: 0.01795697608628334
FOLD: 5, EPOCH: 37, valid_loss: 0.016362425353791978
FOLD: 5, EPOCH: 38, train_loss: 0.01791756769689994
FOLD: 5, EPOCH: 38, valid_loss: 0.016363999702864222
FOLD: 5, EPOCH: 39, train_loss: 0.01787870914603655
FOLD: 5, EPOCH: 39, valid_loss: 0.016355460406177573
FOLD: 5, EPOCH: 40, train_loss: 0.017796520406428058
FOLD: 5, EPOCH: 40, valid_loss: 0.016318774575160608
FOLD: 5, EPOCH: 41, train_loss: 0.017756025808361862
FOLD: 5, EPOCH: 41, valid_loss: 0.016307180023027792
FOLD: 5, EPOCH: 42, train_loss: 0.017653078748247564
FOLD: 5, EPOCH: 42, valid_loss: 0.016246395289070077
FOLD: 5, EPOCH: 43, train_loss: 0.017605714213389616
FOLD: 5, EPOCH: 43, valid_loss: 0.01627288469009929
FOLD: 5, EPOCH: 44, train_loss: 0.017584558695745774
FOLD: 5, EPOCH: 44, valid_loss: 0.016247208851079147
FOLD: 5, EPOCH: 45, train_loss: 0.017535107066998117
FOLD: 5, EPOCH: 45, valid_loss: 0.016265810156861942
FOLD: 5, EPOCH: 46, train_loss: 0.0174877368009243

FOLD: 7, EPOCH: 15, train_loss: 0.019916993422577016
FOLD: 7, EPOCH: 15, valid_loss: 0.016846197553806834
FOLD: 7, EPOCH: 16, train_loss: 0.019750573767874487
FOLD: 7, EPOCH: 16, valid_loss: 0.016957732952303357
FOLD: 7, EPOCH: 17, train_loss: 0.019784743658816203
FOLD: 7, EPOCH: 17, valid_loss: 0.01727453867594401
FOLD: 7, EPOCH: 18, train_loss: 0.019803267378264513
FOLD: 7, EPOCH: 18, valid_loss: 0.01728059434228473
FOLD: 7, EPOCH: 19, train_loss: 0.01979201291807187
FOLD: 7, EPOCH: 19, valid_loss: 0.01740066231124931
FOLD: 7, EPOCH: 20, train_loss: 0.019706446152084913
FOLD: 7, EPOCH: 20, valid_loss: 0.01707116286787722
FOLD: 7, EPOCH: 21, train_loss: 0.019709920558409814
FOLD: 7, EPOCH: 21, valid_loss: 0.01723966457777553
FOLD: 7, EPOCH: 22, train_loss: 0.019651895938202355
FOLD: 7, EPOCH: 22, valid_loss: 0.01709723700251844
FOLD: 7, EPOCH: 23, train_loss: 0.019583467203073014
FOLD: 7, EPOCH: 23, valid_loss: 0.01684561289019055
FOLD: 7, EPOCH: 24, train_loss: 0.019630059814797
FOLD

FOLD: 8, EPOCH: 43, train_loss: 0.01754607682904372
FOLD: 8, EPOCH: 43, valid_loss: 0.01606588324324952
FOLD: 8, EPOCH: 44, train_loss: 0.017554293864239484
FOLD: 8, EPOCH: 44, valid_loss: 0.016035097651183605
FOLD: 8, EPOCH: 45, train_loss: 0.017440882845757864
FOLD: 8, EPOCH: 45, valid_loss: 0.016042838080061808
FOLD: 8, EPOCH: 46, train_loss: 0.017461552881659605
FOLD: 8, EPOCH: 46, valid_loss: 0.015994175130294427
FOLD: 8, EPOCH: 47, train_loss: 0.017391681026380796
FOLD: 8, EPOCH: 47, valid_loss: 0.01599874461276664
FOLD: 8, EPOCH: 48, train_loss: 0.017324827718906678
FOLD: 8, EPOCH: 48, valid_loss: 0.01602091133180592
FOLD: 8, EPOCH: 49, train_loss: 0.017428237753800858
FOLD: 8, EPOCH: 49, valid_loss: 0.015998558348251715
FOLD: 9, EPOCH: 0, train_loss: 0.7079055217596201
FOLD: 9, EPOCH: 0, valid_loss: 0.5995103650622897
FOLD: 9, EPOCH: 1, train_loss: 0.1856328937678765
FOLD: 9, EPOCH: 1, valid_loss: 0.02335134893655777
FOLD: 9, EPOCH: 2, train_loss: 0.0239836586973606
FOLD: 9, EP

In [37]:
for i in range(len(target_cols)):
    fea=target_cols[i]
    test[fea]=predictions[:,i]

In [38]:
valid_results = train_targets_scored.drop(columns=target_cols).merge(train[['sig_id']+target_cols], on='sig_id', how='left').fillna(0)

y_true = train_targets_scored[target_cols].values
y_pred = valid_results[target_cols].values

score = 0
for i in range(len(target_cols)):
    score_ = log_loss(y_true[:, i], y_pred[:, i])
    score += score_ / target.shape[1]
    
print("CV log_loss: ", score)

CV log_loss:  0.014421569885757423


In [ ]:
sub = sample_submission.drop(columns=target_cols).merge(test[['sig_id']+target_cols], on='sig_id', how='left').fillna(0)
sub.to_csv('submission.csv', index=False)